Based on the video and our discussion, here are your formatted notes on Caching and Persistence in Apache Spark.

### **Apache Spark Optimization: Caching & Persistence**

**1. The Core Concept**
*   **Definition:** Caching is an optimization technique that stores a DataFrame in **Storage Memory (RAM)** so Spark does not have to recompute it every time it is used.
*   **The Problem (Lazy Evaluation):** Because Spark is lazily evaluated, if you create `DF1` and use it to create `DF2` and `DF3`, Spark will re-run the code to generate `DF1` from scratch for *each* subsequent action,.
*   **The Solution:** Calling `.cache()` or `.persist()` computes the DataFrame once and saves it. Subsequent steps use a fast **"In-memory table scan"** instead of re-running the entire plan.

**2. Cache vs. Persist**
*   **`.cache()` (The Shortcut):**
    *   A wrapper for `.persist()`.
    *   **Default Behavior:** Uses `MEMORY_AND_DISK`. It tries to fit data in RAM; if full, it spills the excess to the disk.
    *   **Best For:** Quick, default optimization without complex configuration.
*   **`.persist()` (The Flexible Option):**
    *   Allows you to define exactly *where* and *how* data is stored using specific **Storage Levels**.

**3. Common Storage Levels**
*   **`MEMORY_AND_DISK` (Default):** The safest option. Stores in RAM; spills to disk if memory is full.
*   **`MEMORY_ONLY`:** Stores only in RAM. If it doesn't fit, Spark will **recompute** the missing partitions when needed (risky for expensive calculations).
*   **`DISK_ONLY`:** Stores only on the hard drive. Slower than RAM, but does not consume execution memory.

**4. Cleaning Up**
*   **`.unpersist()`:** Removes the DataFrame from memory/disk to free up resources for other tasks.

***

In [0]:
df1 = spark.read.table('testdb.testschema.healthcare_dataset')

display(df1)

In [0]:
df2 = df1.groupBy('Name').count()

display(df2)

In [0]:
df2.explain()

In [0]:
df1.cache()

In [0]:
df3 = df1.filter(df1['Age'] > 50)

display(df3)

In [0]:
df3.explain()

In [0]:
from pyspark.storagelevel import StorageLevel

In [0]:
df1.persist(StorageLevel.MEMORY_ONLY)

In [0]:
df3 = df1.select('name', 'age')

In [0]:
display(df3)

In [0]:
df3.explain()

In [0]:
df1.unpersist()